# RCA Investigation and Context Publication
Plan allowlisted diagnostics, inspect generated SQL and evidence, produce bounded RCA, and create approval items for reusable knowledge.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import pandas as pd
import yaml

ROOT = Path.cwd()
if not (ROOT / 'config').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from dq_agent.config import TablePair, load_app_config
from dq_agent.context_store import read_context, write_context_proposals
from dq_agent.context_utils import build_context_proposals, configure_workflow_logging, logged_step, workflow_paths, write_approval_workbook
from dq_agent.measures import (
    default_diagnostic_requests, diagnostic_sql, evidence_based_rca,
    load_measure_configuration, rca_context_record, select_reconciliation_groups,
    validate_diagnostic_request,
)
from dq_agent.query_engine import QueryGuard
from dq_agent.reporting import write_json, write_measure_reports

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RECONCILIATION_RUN_ID = None
config = load_app_config(ROOT)
paths = workflow_paths(config, RUN_ID, 'rca')
logger = configure_workflow_logging(paths['log'], config.project.log_level)
measure_config = load_measure_configuration(config)
guard = QueryGuard()
sample = yaml.safe_load((ROOT / 'inputs/measure_sample_metadata.yaml').read_text(encoding='utf-8'))
reconciliation_root = config.path(config.project.outputs_dir) / 'reconciliation'
runs = sorted(reconciliation_root.glob('*/resolved_measures.json'), key=lambda path: path.stat().st_mtime)
if RECONCILIATION_RUN_ID:
    measure_file = reconciliation_root / RECONCILIATION_RUN_ID / 'resolved_measures.json'
else:
    if not runs: raise FileNotFoundError('Run 06_measure_reconciliation.ipynb first')
    measure_file = runs[-1]
source_run = measure_file.parent
measures = json.loads(measure_file.read_text(encoding='utf-8'))
results = json.loads((source_run / 'measure_reconciliation.json').read_text(encoding='utf-8'))
failures = [row for row in results if row.get('status') == 'FAIL']
print('Source run:', source_run.name, 'Failures:', len(failures))

In [ ]:
pairs = {
    'sample_fact_sales': TablePair(pair_id='sample_fact_sales', mode='migration', source_catalog='main', source_schema='sales', source_table='fact_sales', target_project='your-gcp-project', target_dataset='analytics', target_table='fact_sales'),
    'sample_fact_orders': TablePair(pair_id='sample_fact_orders', mode='migration', source_catalog='main', source_schema='sales', source_table='fact_orders', target_project='your-gcp-project', target_dataset='analytics', target_table='fact_orders'),
}
mappings = {
    'sample_fact_sales': [
        {'source_column': 'sales_date', 'target_column': 'sales_date'},
        {'source_column': 'brand_id', 'target_column': 'brand_sid'},
        {'source_column': 'market_id', 'target_column': 'market_sid'},
        {'source_column': 'source_system', 'target_column': 'source_system'},
    ],
    'sample_fact_orders': [{'source_column': 'channel_id', 'target_column': 'channel_sid'}],
}
measure_by_id = {(row['pair_id'], row['measure_id']): row for row in measures}

In [ ]:
with logged_step(logger, paths['checkpoint'], 'PLAN_RCA_DIAGNOSTIC'):
    plans = []
    plan_keys = set()
    groups_by_measure = {}
    for failure in failures:
        key = (failure['pair_id'], failure['measure_id'])
        measure = measure_by_id[key]
        pair = pairs[failure['pair_id']]
        groups = select_reconciliation_groups(measure, sample['tables'][pair.pair_id], mappings[pair.pair_id], measure_config.settings.max_groupings_per_measure)
        groups_by_measure[key] = groups
        requests = default_diagnostic_requests(measure, groups, measure_config.diagnostics)
        for request in requests:
            validate_diagnostic_request(request, measure, groups, measure_config.diagnostics)
            plan_key = (pair.pair_id, measure['measure_id'], request.query_type, request.grouping_id, request.granularity)
            if plan_key not in plan_keys:
                plans.append({'pair_id': pair.pair_id, 'measure_id': measure['measure_id'], **request.model_dump()})
                plan_keys.add(plan_key)
                logger.info('PLAN_RCA_DIAGNOSTIC pair=%s measure=%s request=%s', pair.pair_id, measure['measure_id'], request.model_dump())
plans_frame = pd.DataFrame(plans)
display(plans_frame)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'VALIDATE_DIAGNOSTIC_REQUEST'):
    generated = []
    for plan in plans:
        pair = pairs[plan['pair_id']]
        measure = measure_by_id[(pair.pair_id, plan['measure_id'])]
        request = next(item for item in default_diagnostic_requests(measure, groups_by_measure[(pair.pair_id, measure['measure_id'])], measure_config.diagnostics) if item.query_type == plan['query_type'])
        for side in ('source', 'target'):
            sql = diagnostic_sql(request, measure, pair, side, groups_by_measure[(pair.pair_id, measure['measure_id'])], [], measure_config.settings.maximum_group_cardinality)
            allowed = {pair.source_name} if side == 'source' else {pair.target_name}
            checked = guard.validate(sql, 'databricks' if side == 'source' else 'bigquery', allowed)
            sql_file = paths['output'] / 'sql' / f"{pair.pair_id}_{measure['measure_id']}_{request.query_type}_{side}.sql"
            sql_file.parent.mkdir(parents=True, exist_ok=True)
            sql_file.write_text(checked + '\n', encoding='utf-8')
            generated.append({'pair_id': pair.pair_id, 'measure_id': measure['measure_id'], 'query_type': request.query_type, 'side': side, 'sql_file': str(sql_file), 'sql': checked})
display(pd.DataFrame(generated))

In [ ]:
with logged_step(logger, paths['checkpoint'], 'EXECUTE_DIAGNOSTIC_QUERY'):
    evidence_by_failure = {}
    available = sample.get('diagnostic_results', {})
    for failure in failures:
        pair_id, measure_id = failure['pair_id'], failure['measure_id']
        evidence = []
        for plan in [row for row in plans if row['pair_id'] == pair_id and row['measure_id'] == measure_id]:
            prefix = f"{pair_id}|{measure_id}|{plan['query_type']}"
            if prefix + '|source' not in available or prefix + '|target' not in available:
                continue
            evidence.append({
                'request': {key: plan.get(key) for key in ('query_type', 'purpose', 'grouping_id', 'granularity')},
                'source': available[prefix + '|source'], 'target': available[prefix + '|target'],
                'query_references': {
                    'source': str(paths['output'] / 'sql' / f"{pair_id}_{measure_id}_{plan['query_type']}_source.sql"),
                    'target': str(paths['output'] / 'sql' / f"{pair_id}_{measure_id}_{plan['query_type']}_target.sql"),
                },
            })
        evidence_by_failure[failure['rule_id']] = evidence
        logger.info('EXECUTE_DIAGNOSTIC_QUERY pair=%s measure=%s evidence_sets=%s', pair_id, measure_id, len(evidence))

In [ ]:
with logged_step(logger, paths['checkpoint'], 'GENERATE_EVIDENCE_BASED_RCA'):
    rca_records = []
    for failure in failures:
        rca = evidence_based_rca(failure, evidence_by_failure[failure['rule_id']])
        rca.update({'pair_id': failure['pair_id'], 'rule_id': failure['rule_id'], 'diagnostics': evidence_by_failure[failure['rule_id']]})
        rca_records.append(rca)
        logger.info('GENERATE_EVIDENCE_BASED_RCA pair=%s measure=%s classification=%s confidence=%s evidence=%s', failure['pair_id'], failure['measure_id'], rca['classification'], rca['confidence'], rca['evidence_observed'])
display(pd.json_normalize(rca_records, sep='.'))

In [ ]:
with logged_step(logger, paths['checkpoint'], 'CREATE_APPROVAL_ITEM'):
    trusted = read_context(config, logger=logger) if config.project.context_store.enabled else pd.DataFrame()
    reusable = [row for row in rca_records if row.get('reusable_candidate')]
    records = pd.DataFrame([rca_context_record(row, pairs[row['pair_id']], config, RUN_ID) for row in reusable])
    proposals = build_context_proposals(records, trusted, RUN_ID) if not records.empty else pd.DataFrame()
    approval_file = None
    if not proposals.empty:
        if config.project.context_store.enabled:
            write_context_proposals(config, proposals, logger)
            approval_file = paths['pending'] / f'rca_context_approvals_{RUN_ID}.xlsx'
        else:
            approval_file = paths['output'] / f'rca_context_approval_preview_{RUN_ID}.xlsx'
        write_approval_workbook(approval_file, proposals)
        logger.info('CREATE_APPROVAL_ITEM reusable=%s output=%s', len(proposals), approval_file)
print('Approval file:', approval_file)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'WRITE_RCA_RESULTS'):
    report_paths = write_measure_reports(paths['output'], measures, results, rca_records)
    write_json(paths['output'] / 'diagnostic_plans.json', plans)
    logger.info('WRITE_RCA_RESULTS rca=%s outputs=%s', len(rca_records), report_paths)
report_paths

Only reusable RCA candidates are proposed for context. Review a real pending workbook, then use `02_approval_processing.ipynb` to publish it. Pending and rejected records remain outside trusted context.